In [ ]:
# Import all the necessary modules for the Red Pitaya
import numpy as np
from rp.core import RedPitayaBoard
from rp.constants import *
from rp.structs import *
from rp.ram.config import *
from rp.bram.generate import *
from rp.dac.config import *
from rp.adc.config import *
from rp.tuning.lut import LUT
from rp.tuning.values import *
from rp.calib_parameters import *

# Plot settings
from matplotlib import pyplot as plt
%matplotlib inline
plt.rcParams['grid.linestyle'] = '--'  # Grid line style
plt.rcParams['grid.linewidth'] = 0.5  # Grid line width
plt.rcParams['axes.titlesize'] = 14  # Title font size
plt.rcParams['axes.labelsize'] = 12  # Axis label font size
plt.rcParams['xtick.labelsize'] = 10  # X tick label font size
plt.rcParams['ytick.labelsize'] = 10  # Y tick label font size
plt.rcParams['legend.fontsize'] = 10  # Legend font size
plt.rcParams['xtick.direction'] = 'in'  # X tick direction
plt.rcParams['ytick.direction'] = 'in'  # Y tick direction
plt.rcParams['xtick.top'] = True  # Show ticks on top
plt.rcParams['ytick.right'] = True


def rp_con():
    # Init RedPitayaBoard
    pita = RedPitayaBoard(verbose=False)
    # Init RAM with 8MB size
    ramCfg = get_ram_config(RAM_SIZE.MB_8)
    pita.sendConfigParams(ramCfg, RAM_INIT_CONFIG_ID)
    return pita, ramCfg
    
def rp_close(pita):
    pita.close()

Generate multiple signals types

In [ ]:
####################################################################################################
pita, ramCfg = rp_con()
####################################################################################################

# Internal BRAM can store a max of 16384 samples, num_samples variable limits 
# the number of samples written to the BRAM, the rest will be filled with zeros
bram_max_sample = 16384
num_samples = bram_max_sample

gen_funcs = (gen_sine_cosine, gen_sawtooth, gen_triangle)
num_funcs = 3
name = ('Sine', 'Sawtooth', 'Triangle')
I = []

for i in range(len(gen_funcs)):

    # Create a BRAM config for internal BRAM, unsused BRAM area are filled with zeros
    bramCfg = get_bram_struct(
        gen_func = gen_funcs[i],
        samples = num_samples, 
        bit_range = 14,
        phase = 0,
        amplitude_div = 1,
    )
    # Fill BRAMs with data
    pita.sendConfigParams(bramCfg, DUMMY_DATA_GEN_BRAM_ID)

    # Route Dummy Data Generator's debug port to RAM Writer
    pita.sendCommand(SELECT_RAM_MUX_INPUT, RAM_MUX_IN_DUMMY_DEBUG)

    # Dummy Data Generator config:
    dummyCfg = DummyDataGenConfig(
        mode = 0b111,                # mode = debug_mode | bram_mode | continuous mode
        max_samples = 0,             # Ignored in continuous mode
        sample_delay = 1,            # Full speed
        max_cnt = 0                  # Ignored in BRAM mode
    )
    pita.sendConfigParams(dummyCfg, DUMMY_DATA_GEN_CONFIG_ID)

    # Start LIA-Mixer test and receive data
    pita.lia_debug()

    # Receive data from RP and extract only lower 16 bits (channel 1)
    data = pita.receive_adc_data_package(ramCfg.ram_size_bytes)
    data = np.frombuffer(data, dtype=np.int32)
    i = np.vectorize(twos_to_int)(data & 0xFFFF, 16)
    I.append(i)
    
####################################################################################################
rp_close(pita)
####################################################################################################

plt.figure(figsize=(12, 6))
for i in range(num_funcs):
    plt.plot(I[i][:2*bram_max_sample], label=name[i])
    
plt.xlabel('Sample (n)')
plt.ylabel('Amplitude (count)')
plt.legend()

plt.savefig('dummy_gen_func.png', bbox_inches='tight')

In [ ]:
####################################################################################################
pita, ramCfg = rp_con()
####################################################################################################

# Dummy Data Generator config:
dummyCfg = DummyDataGenConfig(
    mode = 0b111,                # mode = debug_mode | bram_mode | continuous mode
    max_samples = 0,             # Ignored in continuous mode
    sample_delay = 1,            # Full speed
    max_cnt = 0                  # Ignored in BRAM mode
)
pita.sendConfigParams(dummyCfg, DUMMY_DATA_GEN_CONFIG_ID)

# Amount of samples to generate
num_samples = 16384

phase = np.linspace(0, 2*np.pi, 5, endpoint=False)
output = []

for i in range(len(phase)):

    # Create a BRAM config for internal BRAM, unsused BRAM area are filled with zeros
    bramCfg = get_bram_struct(
        gen_func = gen_sine_cosine,
        samples = num_samples, 
        bit_range = 14,
        phase = phase[i],
        amplitude_div = 1,
    )
    # Fill BRAMs with data
    pita.sendConfigParams(bramCfg, DUMMY_DATA_GEN_BRAM_ID)

    # Route Dummy Data Generator's debug port to RAM Writer
    pita.sendCommand(SELECT_RAM_MUX_INPUT, RAM_MUX_IN_DUMMY_DEBUG)

    # Start LIA-Mixer test and receive data
    pita.lia_debug()

    # Receive data from RP and extract only lower 16 bits (channel 1)
    data = pita.receive_adc_data_package(ramCfg.ram_size_bytes)
    data = np.frombuffer(data, dtype=np.int32)
    data = np.vectorize(twos_to_int)(data & 0xFFFF, 16)
    output.append(data)
    
####################################################################################################
rp_close(pita)
####################################################################################################

plt.figure(figsize=(12, 6))
for i in range(len(phase)):
    plt.plot(output[i][:2*num_samples], label=f'Phase = {phase[i]:.2f}')
    
plt.xlabel('Sample (n)')
plt.ylabel('Amplitude (count)')
plt.legend()